In [0]:
from pyspark.sql.functions import col,when
import pyspark.sql.functions as f
from pyspark.sql.window import Window
import dlt

In [0]:
# TODO change all int field to float fields

In [0]:
source_schema = spark.conf.get("source_schema")

catalog = spark.conf.get("pipelines.catalog")
schema = spark.conf.get("pipelines.schema")

In [0]:
import dlt
from pyspark.sql.functions import col, expr

@dlt.view
def adm_updates(temporary=True):
  return (spark.readStream
    .option("readChangeFeed", "true")
    .table(f'{source_schema}.admissions')
    .filter(col('_change_type') != 'update_preimage')
    .withColumnRenamed('_commit_timestamp', 'commit_timestamp')
    # .drop('_change_type')
    .drop('_commit_version')
    #  admission_type features
    .withColumn("admission_type_direct_observation", when(col('admission_type') == "DIRECT OBSERVATION",1).otherwise(0))
    .withColumn("admission_type_eu_observation", when(col('admission_type') == "EU OBSERVATION",1).otherwise(0))
    .withColumn("admission_type_ew_emer", when(col('admission_type') == "EW EMER.",1).otherwise(0))
    .withColumn("admission_type_elective", when(col('admission_type') == "ELECTIVE",1).otherwise(0))
    .withColumn("admission_type_surgical_same_day_admission", when(col('admission_type') == "SURGICAL SAME DAY ADMISSION",1).otherwise(0))
    .withColumn("admission_type_observation_admit", when(col('admission_type') == "OBSERVATION ADMIT",1).otherwise(0))
    .withColumn("admission_type_ambulatory_observation", when(col('admission_type') == "AMBULATORY OBSERVATION",1).otherwise(0))
    .withColumn("admission_type_direct_emer", when(col('admission_type') == "DIRECT EMER.",1).otherwise(0))
    .withColumn("admission_type_urgent", when(col('admission_type') == "URGENT",1).otherwise(0))
    .drop("admission_type")
    # admission_location features
    .withColumn("admission_location_internal_transfer_to_or_from_psych", when(col('admission_location') == "INTERNAL TRANSFER TO OR FROM PSYCH", 1).otherwise(0))
    .withColumn("admission_location_procedure_site", when(col('admission_location') == "PROCEDURE SITE", 1).otherwise(0))
    .withColumn("admission_location_emergency_room", when(col('admission_location') == "EMERGENCY ROOM", 1).otherwise(0))
    .withColumn("admission_location_physician_referral", when(col('admission_location') == "PHYSICIAN REFERRAL", 1).otherwise(0))
    .withColumn("admission_location_transfer_from_skilled_nursing_facility", when(col('admission_location') == "TRANSFER FROM SKILLED NURSING FACILITY", 1).otherwise(0))
    .withColumn("admission_location_walk_in_self_referral", when(col('admission_location') == "WALK-IN/SELF REFERRAL", 1).otherwise(0))
    .withColumn("admission_location_clinic_referral", when(col('admission_location') == "CLINIC REFERRAL", 1).otherwise(0))
    .withColumn("admission_location_pacu", when(col('admission_location') == "PACU", 1).otherwise(0))
    .withColumn("admission_location_transfer_from_hospital", when(col('admission_location') == "TRANSFER FROM HOSPITAL", 1).otherwise(0))
    .withColumn("admission_location_information_not_available", when(col('admission_location') == "INFORMATION NOT AVAILABLE", 1).otherwise(0))
    .withColumn("admission_location_ambulatory_surgery_transfer", when(col('admission_location') == "AMBULATORY SURGERY TRANSFER", 1).otherwise(0))
    .drop("admission_location")
    # insurance features
    .withColumn("insurance_private", when(col('insurance') == "Private", 1).otherwise(0))
    .withColumn("insurance_other", when(col('insurance') == "Other", 1).otherwise(0))
    .withColumn("insurance_medicaid", when(col('insurance') == "Medicaid", 1).otherwise(0))
    .withColumn("insurance_no_charge", when(col('insurance') == "No charge", 1).otherwise(0))
    .withColumn("insurance_medicare", when(col('insurance') == "Medicare", 1).otherwise(0))
    .withColumn("insurance_none", when(col('insurance').isNull(), 1).otherwise(0))
    .drop("insurance")
    # marital_status features
    .withColumn("marital_status_widowed", when(col('marital_status') == "WIDOWED", 1).otherwise(0))
    .withColumn("marital_status_single", when(col('marital_status') == "SINGLE", 1).otherwise(0))
    .withColumn("marital_status_married", when(col('marital_status') == "MARRIED", 1).otherwise(0))
    .withColumn("marital_status_divorced", when(col('marital_status') == "DIVORCED", 1).otherwise(0))
    .withColumn("marital_status_none", when(col('marital_status').isNull(), 1).otherwise(0))
    .drop("marital_status")
    .select(
      "hadm_id",
      "admission_type_direct_observation",
      "admission_type_eu_observation",
      "admission_type_ew_emer",
      "admission_type_elective",
      "admission_type_surgical_same_day_admission",
      "admission_type_observation_admit",
      "admission_type_ambulatory_observation",
      "admission_type_direct_emer",
      "admission_type_urgent",
      "admission_location_internal_transfer_to_or_from_psych",
      "admission_location_procedure_site",
      "admission_location_emergency_room",
      "admission_location_physician_referral",
      "admission_location_transfer_from_skilled_nursing_facility",
      "admission_location_walk_in_self_referral",
      "admission_location_clinic_referral",
      "admission_location_pacu",
      "admission_location_transfer_from_hospital",
      "admission_location_information_not_available",
      "admission_location_ambulatory_surgery_transfer",
      "insurance_private",
      "insurance_other",
      "insurance_medicaid",
      "insurance_no_charge",
      "insurance_medicare",
      "insurance_none",
      "marital_status_widowed",
      "marital_status_single",
      "marital_status_married",
      "marital_status_divorced",
      "marital_status_none",
      "_change_type",
      "commit_timestamp"
    )
    )

In [0]:
dlt.create_streaming_table(
  "admissions_features",
  schema="""
  hadm_id INT NOT NULL PRIMARY KEY,
  admission_type_direct_observation INT,
  admission_type_eu_observation INT,
  admission_type_ew_emer INT,
  admission_type_elective INT,
  admission_type_surgical_same_day_admission INT,
  admission_type_observation_admit INT,
  admission_type_ambulatory_observation INT,
  admission_type_direct_emer INT,
  admission_type_urgent INT,
  admission_location_internal_transfer_to_or_from_psych INT,
  admission_location_procedure_site INT,
  admission_location_emergency_room INT,
  admission_location_physician_referral INT,
  admission_location_transfer_from_skilled_nursing_facility INT,
  admission_location_walk_in_self_referral INT,
  admission_location_clinic_referral INT,
  admission_location_pacu INT,
  admission_location_transfer_from_hospital INT,
  admission_location_information_not_available INT,
  admission_location_ambulatory_surgery_transfer INT,
  insurance_private INT,
  insurance_other INT,
  insurance_medicaid INT,
  insurance_no_charge INT,
  insurance_medicare INT,
  insurance_none INT,
  marital_status_widowed INT,
  marital_status_single INT,
  marital_status_married INT,
  marital_status_divorced INT,
  marital_status_none INT
                           """)

dlt.create_auto_cdc_flow(
  target = "admissions_features",
  source = "adm_updates",
  keys = ["hadm_id"],
  sequence_by = col("commit_timestamp"),
  apply_as_deletes = expr("_change_type = 'delete'"),
  # apply_as_truncates = expr("operation = 'TRUNCATE'"),
  except_column_list = ["_change_type", "commit_timestamp"],
  stored_as_scd_type = 1
)

In [0]:
# dlt.create_streaming_table(
#   "admissions_features",
#   schema="""
#   subject_id INT,
#   hadm_id INT NOT NULL PRIMARY KEY,
#   admittime TIMESTAMP,
#   dischtime TIMESTAMP,
#   deathtime TIMESTAMP,
#   admit_provider_id STRING,
#   discharge_location STRING,
#   language STRING,
#   race STRING,
#   edregtime TIMESTAMP,
#   edouttime TIMESTAMP,
#   hospital_expire_flag INT,
#   admission_type_direct_observation INT,
#   admission_type_eu_observation INT,
#   admission_type_ew_emer INT,
#   admission_type_elective INT,
#   admission_type_surgical_same_day_admission INT,
#   admission_type_observation_admit INT,
#   admission_type_ambulatory_observation INT,
#   admission_type_direct_emer INT,
#   admission_type_urgent INT,
#   admission_location_internal_transfer_to_or_from_psych INT,
#   admission_location_procedure_site INT,
#   admission_location_emergency_room INT,
#   admission_location_physician_referral INT,
#   admission_location_transfer_from_skilled_nursing_facility INT,
#   admission_location_walk_in_self_referral INT,
#   admission_location_clinic_referral INT,
#   admission_location_pacu INT,
#   admission_location_transfer_from_hospital INT,
#   admission_location_information_not_available INT,
#   admission_location_ambulatory_surgery_transfer INT,
#   insurance_private INT,
#   insurance_other INT,
#   insurance_medicaid INT,
#   insurance_no_charge INT,
#   insurance_medicare INT,
#   insurance_none INT,
#   marital_status_widowed INT,
#   marital_status_single INT,
#   marital_status_married INT,
#   marital_status_divorced INT,
#   marital_status_none INT
#                            """)

# dlt.create_auto_cdc_flow(
#   target = "admissions_features",
#   source = "adm_updates",
#   keys = ["hadm_id"],
#   sequence_by = col("commit_timestamp"),
#   apply_as_deletes = expr("_change_type = 'delete'"),
#   # apply_as_truncates = expr("operation = 'TRUNCATE'"),
#   except_column_list = ["_change_type", "commit_timestamp"],
#   stored_as_scd_type = 1
# )

In [0]:
# import dlt
# from pyspark.sql import functions as f, Window
# from pyspark.sql.functions import col

# @dlt.table(
#     name="historic_admissions_features",
#     schema="""
#   subject_id INT,
#   hadm_id INT NOT NULL PRIMARY KEY,
#   admittime TIMESTAMP,
#   dischtime TIMESTAMP,
#   deathtime TIMESTAMP,
#   admission_type STRING,
#   admit_provider_id STRING,
#   admission_location STRING,
#   discharge_location STRING,
#   insurance STRING,
#   language STRING,
#   marital_status STRING,
#   race STRING,
#   edregtime TIMESTAMP,
#   edouttime TIMESTAMP,
#   hospital_expire_flag INT,
#   new_patient INT,
#   IS_A_READMISSION DOUBLE,
#   `30_DAY_READMISSION` DOUBLE,
#   `30_DAY_READMISSION_6_months` DOUBLE,
#   `30_DAY_READMISSION_12_months` DOUBLE,
#   prev_admissions_6_months BIGINT,
#   prev_admissions_12_months BIGINT""")
# def historic_admissions_features():
#     w = Window.partitionBy("subject_id").orderBy(col("admittime"))
#     adm = spark.table(f'{source_schema}.admissions')
#     # noramlly you'd choose today, but this demo happens at various points in history, so we fetch "today" from the max admission date
#     # max_adm_date = adm.select(f.max(col('admittime'))).collect()[0][0]
#     return (
#         spark.table(f'{source_schema}.admissions')
#         # .filter(col('admittime') <= max_adm_date - f.make_inverval(years=3))
#         .withColumn('last_discharge', f.lag(f.col('dischtime')).over(w))
#         .withColumn('new_patient', f.when(f.col('last_discharge').isNull(), 1).otherwise(0))
#         .withColumn('IS_A_READMISSION', f.when(
#             f.col('last_discharge') > f.date_trunc('dd', f.col('admittime')) - f.expr('INTERVAL 30 DAYS'), 1
#         ).otherwise(0).cast('DOUBLE'))
#         .withColumn('30_DAY_READMISSION', f.coalesce(f.lead('IS_A_READMISSION').over(w), f.lit(0)))
#         .withColumn('30_DAY_READMISSION_6_months', f.sum(col('IS_A_READMISSION')).over(
#             Window.partitionBy("subject_id").orderBy(col("admittime").cast("long")).rangeBetween(-60*60*24*180, 0)
#         ))
#         .withColumn('30_DAY_READMISSION_12_months', f.sum(col('IS_A_READMISSION')).over(
#             Window.partitionBy("subject_id").orderBy(col("admittime").cast("long")).rangeBetween(-60*60*24*365, 0)
#         ))
#         .withColumn('prev_admissions_6_months', f.count(col('admittime')).over(
#             Window.partitionBy("subject_id").orderBy(col("admittime").cast("long")).rangeBetween(-60*60*24*180, 0)
#         ))
#         .withColumn('prev_admissions_12_months', f.count(col('admittime')).over(
#             Window.partitionBy("subject_id").orderBy(col("admittime").cast("long")).rangeBetween(-60*60*24*365, 0)
#         ))
#         .drop('last_discharge')
#     )

In [0]:
import dlt
from pyspark.sql import functions as f, Window
from pyspark.sql.functions import col

@dlt.table(
    name="historic_admissions_features",
    schema="""
  hadm_id INT NOT NULL PRIMARY KEY,
  new_patient INT,
  IS_A_READMISSION DOUBLE,
  `30_DAY_READMISSION` DOUBLE,
  `30_DAY_READMISSION_6_months` DOUBLE,
  `30_DAY_READMISSION_12_months` DOUBLE,
  prev_admissions_6_months BIGINT,
  prev_admissions_12_months BIGINT""")
def historic_admissions_features():
    w = Window.partitionBy("subject_id").orderBy(col("admittime"))
    adm = spark.table(f'{source_schema}.admissions')
    # noramlly you'd choose today, but this demo happens at various points in history, so we fetch "today" from the max admission date
    # max_adm_date = adm.select(f.max(col('admittime'))).collect()[0][0]
    return (
        spark.table(f'{source_schema}.admissions')
        # .filter(col('admittime') <= max_adm_date - f.make_inverval(years=3))
        .withColumn('last_discharge', f.lag(f.col('dischtime')).over(w))
        .withColumn('new_patient', f.when(f.col('last_discharge').isNull(), 1).otherwise(0))
        .withColumn('IS_A_READMISSION', f.when(
            f.col('last_discharge') > f.date_trunc('dd', f.col('admittime')) - f.expr('INTERVAL 30 DAYS'), 1
        ).otherwise(0).cast('DOUBLE'))
        .withColumn('30_DAY_READMISSION', f.coalesce(f.lead('IS_A_READMISSION').over(w), f.lit(0)))
        .withColumn('30_DAY_READMISSION_6_months', f.sum(col('IS_A_READMISSION')).over(
            Window.partitionBy("subject_id").orderBy(col("admittime").cast("long")).rangeBetween(-60*60*24*180, 0)
        ))
        .withColumn('30_DAY_READMISSION_12_months', f.sum(col('IS_A_READMISSION')).over(
            Window.partitionBy("subject_id").orderBy(col("admittime").cast("long")).rangeBetween(-60*60*24*365, 0)
        ))
        .withColumn('prev_admissions_6_months', f.count(col('admittime')).over(
            Window.partitionBy("subject_id").orderBy(col("admittime").cast("long")).rangeBetween(-60*60*24*180, 0)
        ))
        .withColumn('prev_admissions_12_months', f.count(col('admittime')).over(
            Window.partitionBy("subject_id").orderBy(col("admittime").cast("long")).rangeBetween(-60*60*24*365, 0)
        ))
        .select(
            "hadm_id",
            "new_patient",
            "IS_A_READMISSION",
            "30_DAY_READMISSION",
            "30_DAY_READMISSION_6_months",
            "30_DAY_READMISSION_12_months",
            "prev_admissions_6_months",
            "prev_admissions_12_months",
        )
    )

In [0]:
import dlt
from pyspark.sql.functions import col, expr

@dlt.view
def pat_updates(temporary=True):
  return (spark.readStream
    .option("readChangeFeed", "true")
    .table(f'{source_schema}.patients')
    .filter(col('_change_type') != 'update_preimage')
    .withColumnRenamed('_commit_timestamp', 'commit_timestamp')
    # .drop('_change_type')
    .drop('_commit_version')

    # Gender Features
    .withColumn("gender_f", when(col('gender') == "F", 1).otherwise(0))
    .withColumn("gender_m", when(col('gender') == "M", 1).otherwise(0))
    .select(
      "subject_id",
      "gender_f",
      "gender_m",
      "_change_type", 
      "commit_timestamp")
  )
   
dlt.create_streaming_table(
  "patient_features",
  schema="""
  subject_id INT NOT NULL PRIMARY KEY,
  gender_f INT,
  gender_m INT"""
  )

dlt.create_auto_cdc_flow(
  target = "patient_features",
  source = "pat_updates",
  keys = ["subject_id"],
  sequence_by = col("commit_timestamp"),
  apply_as_deletes = expr("_change_type = 'delete'"),
  # apply_as_truncates = expr("operation = 'TRUNCATE'"),
  except_column_list = ["_change_type", "commit_timestamp"],
  stored_as_scd_type = 1
)

In [0]:
@dlt.view
def adm_updates2(temporary=True):
  return (spark.readStream
    .option("readChangeFeed", "true")
    .table(f'{source_schema}.admissions')
    .filter(col('_change_type') != 'update_preimage')
    .withColumnRenamed('_commit_timestamp', 'commit_timestamp')
    # .drop('_change_type')
    .drop('_commit_version')
    #  admission_type features
    .join(
      spark.table(f"{source_schema}.patients").select("subject_id", "dob"),
       ["subject_id"], "left")
    .withColumn("age_at_admission", f.datediff(col("admittime"), col("dob")) / 365.25)
    .select("hadm_id", "age_at_admission", "commit_timestamp","_change_type")
  )

In [0]:
dlt.create_streaming_table(
  "age_at_admission",
  schema="""
  hadm_id INT NOT NULL PRIMARY KEY,
  age_at_admission DOUBLE
  """
  )

dlt.create_auto_cdc_flow(
  target = "age_at_admission",
  source = "adm_updates2",
  keys = ["hadm_id"],
  sequence_by = col("commit_timestamp"),
  apply_as_deletes = expr("_change_type = 'delete'"),
  # apply_as_truncates = expr("operation = 'TRUNCATE'"),
  except_column_list = ["_change_type", "commit_timestamp"],
  stored_as_scd_type = 1
)